In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
df = pd.read_csv('../../data/cardekho_dataset.csv', index_col=[0])
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


### Data Cleaning

In [3]:
# Checking the null values
df.info()

<class 'pandas.DataFrame'>
Index: 15411 entries, 0 to 19543
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   car_name           15411 non-null  str    
 1   brand              15411 non-null  str    
 2   model              15411 non-null  str    
 3   vehicle_age        15411 non-null  int64  
 4   km_driven          15411 non-null  int64  
 5   seller_type        15411 non-null  str    
 6   fuel_type          15411 non-null  str    
 7   transmission_type  15411 non-null  str    
 8   mileage            15411 non-null  float64
 9   engine             15411 non-null  int64  
 10  max_power          15411 non-null  float64
 11  seats              15411 non-null  int64  
 12  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(5), str(6)
memory usage: 1.6 MB


In [4]:
df.isna().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
# Remove unnecessary columns
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)

df.columns

Index(['model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type',
       'transmission_type', 'mileage', 'engine', 'max_power', 'seats',
       'selling_price'],
      dtype='str')

In [6]:
df.model.unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [7]:
## Getting all the different type of features
# Number of numeric features
num_features = df.select_dtypes(exclude='object').columns
print(f'Number of Numeric features are {len(num_features)}')

# Number of Categorical features
cat_features = df.select_dtypes(include='object').columns
print(f'Number of Categorical features are {len(cat_features)}')

# Discrete features
discrete_features = [feature for feature in num_features if len(df[feature].unique() <= 25)]
print(f'Number of Discrete features are {len(discrete_features)}')

# Continuous features
continuous_features = [feature for feature in num_features if feature not in discrete_features < 25]
print(f'Number of Continuous features are {len(continuous_features)}')

Number of Numeric features are 7
Number of Categorical features are 4
Number of Discrete features are 7
Number of Continuous features are 0


In [8]:
# Indeoendent and dependent features
from sklearn.model_selection import train_test_split
X = df.drop('selling_price', axis=1)
y = df.selling_price

### Feature Encoding and Scaling

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [10]:
# Create ColumnTransformer with 3 types of transformers
num_features = X.select_dtypes(exclude='object').columns
onehot_columns = ['seller_type', 'fuel_type', 'transmission_type']
# label_encoder_columns = ['model']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')


preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_transformer, onehot_columns),
        ('StandardScalar', numeric_transformer, num_features)
    ], remainder='passthrough'
)

In [11]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OneHotEncoder', ...), ('StandardScalar', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer

In [12]:
X = preprocessor.fit_transform(X)

In [13]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [14]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((12328, 14), (3083, 14), (12328,), (3083,))

### Model Training and Model Selection

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [16]:
# Create a function to Evaluate the Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = root_mean_squared_error(true, predicted)
    r2_square = r2_score(true, predicted)
    
    return mae, mse, rmse, r2_square 
    

In [17]:
# Model Training
models = {
    "linar Regression": LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'K-Neighbors Regressor': KNeighborsRegressor(),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    
    # Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate train and test models
    train_model_mae, train_model_mse, train_model_rmse, train_model_r2 = evaluate_model(y_train, y_train_pred)
    
    test_model_mae, test_model_mse, test_model_rmse, test_model_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model Performance for train set')
    print(f'{train_model_mae = :.4f}')
    print(f'{train_model_mse = :.4f}')
    print(f'{train_model_rmse = :.4f}')
    print(f'{train_model_r2 = :.4f}')
    
    print('---------------------------------------------')
    
    print('Model Performance for test set')
    print(f'{test_model_mae = :.4f}')
    print(f'{test_model_mse = :.4f}')
    print(f'{test_model_rmse = :.4f}')
    print(f'{test_model_r2 = :.4f}')
    
    print(f'{"=" * 45}')

linar Regression
Model Performance for train set
train_model_mae = 268101.6071
train_model_mse = 306756099359.7596
train_model_rmse = 553855.6665
train_model_r2 = 0.6218
---------------------------------------------
Model Performance for test set
test_model_mae = 279618.5794
test_model_mse = 252550062888.5655
test_model_rmse = 502543.5930
test_model_r2 = 0.6645
Lasso
Model Performance for train set
train_model_mae = 268099.2226
train_model_mse = 306756104248.3742
train_model_rmse = 553855.6710
train_model_r2 = 0.6218
---------------------------------------------
Model Performance for test set
test_model_mae = 279614.7461
test_model_mse = 252549134806.7814
test_model_rmse = 502542.6696
test_model_r2 = 0.6645
Ridge
Model Performance for train set
train_model_mae = 268059.8015
train_model_mse = 306756818740.9266
train_model_rmse = 553856.3160
train_model_r2 = 0.6218
---------------------------------------------
Model Performance for test set
test_model_mae = 279557.2169
test_model_mse = 2

Amongst all the model `KNeighborsRegressor` and `RandomForestRegressor` performed well; lets try some Hyperparameter tunning on it.

In [18]:
# Initizlize few oparameter for hyperparamter tunning
knn_params = {'n_neighbors': [2, 3, 10, 20, 40, 50]}
rf_params = {
    'max_depth': [5, 8, 15, None, 10],
    'max_features': [5, 7, 'auto', 8],
    'min_samples_split': [2, 8, 15, 20],
    'n_estimators': [100, 200, 500, 1000]
}

In [19]:
# model list for hyperparamter tunning
randomcv_models = [ 
        ('KNN', KNeighborsRegressor(), knn_params),   
        ('RF', RandomForestRegressor(), rf_params)
]

In [20]:
from sklearn.model_selection import RandomizedSearchCV
from tqdm import tqdm

model_params = {}
for name, model, params in tqdm(randomcv_models):
    random = RandomizedSearchCV(estimator=model,
                                param_distributions=params,
                                n_iter=100,
                                cv=3,
                                verbose=0,
                                n_jobs=-1)
    
    random.fit(X_train, y_train)
    model_params[name] = random.best_params_
    
for model_name in model_params:
    print(f"------------  Best Params for {model_name} ------------")
    print(model_params[model_name])

100%|██████████| 2/2 [01:30<00:00, 45.19s/it]

------------  Best Params for KNN ------------
{'n_neighbors': 10}
------------  Best Params for RF ------------
{'n_estimators': 100, 'min_samples_split': 2, 'max_features': 7, 'max_depth': 15}


In [21]:
models = {
    'Random Forst': RandomForestRegressor(n_estimators=500,
                                           min_samples_split=2,
                                           max_features=8,
                                           max_depth=None),
    
    'K-Neighbors Regressor': KNeighborsRegressor(n_neighbors=10,
                                                 n_jobs=-1)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    
    # Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate train and test models
    train_model_mae, train_model_mse, train_model_rmse, train_model_r2 = evaluate_model(y_train, y_train_pred)
    
    test_model_mae, test_model_mse, test_model_rmse, test_model_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model Performance for train set')
    print(f'{train_model_mae = :.4f}')
    print(f'{train_model_mse = :.4f}')
    print(f'{train_model_rmse = :.4f}')
    print(f'{train_model_r2 = :.4f}')
    
    print('---------------------------------------------')
    
    print('Model Performance for test set')
    print(f'{test_model_mae = :.4f}')
    print(f'{test_model_mse = :.4f}')
    print(f'{test_model_rmse = :.4f}')
    print(f'{test_model_r2 = :.4f}')
    
    print(f'{"=" * 45}')

Random Forst
Model Performance for train set
train_model_mae = 39009.8837
train_model_mse = 14776992837.4692
train_model_rmse = 121560.6550
train_model_r2 = 0.9818
---------------------------------------------
Model Performance for test set
test_model_mae = 97958.4201
test_model_mse = 44608164651.6490
test_model_rmse = 211206.4503
test_model_r2 = 0.9407
K-Neighbors Regressor
Model Performance for train set
train_model_mae = 103453.4312
train_model_mse = 132091850730.8566
train_model_rmse = 363444.4259
train_model_r2 = 0.8371
---------------------------------------------
Model Performance for test set
test_model_mae = 117441.9802
test_model_mse = 69624031915.5449
test_model_rmse = 263863.6616
test_model_r2 = 0.9075
